# Notebook 02: Basic Statistics

**One Sensor, One Year — Edition 1: India Breathes**

Understanding the scale and spread of each fuel type in 2024.
- Summary statistics per fuel
- Distribution shapes (histograms + box plots)
- Which fuels are steady vs volatile?
- How big is each slice of the pie?

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df24 = pd.read_csv('../data/processed/india_2024.csv', parse_dates=['date'])

fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']
fuel_labels = ['Coal', 'Lignite', 'Gas', 'Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']

# Using Earth & Sky palette
palette = {
    'coal': '#3D2B1F', 'lignite': '#6B4226', 'gas': '#4A90D9',
    'nuclear': '#2EC4B6', 'hydro': '#1B4F72', 'wind': '#85C1E9',
    'solar': '#F5B041', 'other_re': '#A569BD',
}

## 1. Summary Statistics Table

All values in **MU/day** (1 MU = 1 GWh).

In [2]:
stats = df24[fuel_cols].describe().T
stats['share_%'] = (df24[fuel_cols].sum() / df24[fuel_cols].sum().sum() * 100).round(1)
stats['annual_TWh'] = (df24[fuel_cols].sum() / 1000).round(1)  # MU * 366 days → TWh (MU=GWh, so sum/1000=TWh)
stats['cv_%'] = (stats['std'] / stats['mean'] * 100).round(1)  # coefficient of variation

# Reorder columns for readability
stats = stats[['mean', 'std', 'cv_%', 'min', '25%', '50%', '75%', 'max', 'share_%', 'annual_TWh']]
stats.columns = ['Mean', 'Std Dev', 'CV %', 'Min', 'Q1', 'Median', 'Q3', 'Max', 'Share %', 'Annual TWh']
stats = stats.round(1)

print('India Grid 2024 — Daily Generation Statistics (MU/day)')
print('=' * 100)
print(stats.to_string())
print(f'\nTotal daily mean: {df24[fuel_cols].sum(axis=1).mean():.0f} MU/day')
print(f'Total annual: {df24[fuel_cols].sum().sum()/1000:.0f} TWh')

India Grid 2024 — Daily Generation Statistics (MU/day)
            Mean  Std Dev  CV %     Min      Q1  Median      Q3     Max  Share %  Annual TWh
coal      3528.7    269.6   7.6  2814.8  3345.7  3541.4  3722.4  4054.0     72.9      1291.5
lignite     93.8     12.8  13.6    50.1    86.0    93.7   103.0   126.2      1.9        34.3
gas         92.5     40.7  44.0    38.0    66.8    81.6   104.0   209.4      1.9        33.9
nuclear    147.7     18.5  12.5   105.8   133.5   149.0   162.6   178.7      3.1        54.0
hydro      391.6    176.6  45.1   172.8   232.5   325.2   509.7   746.9      8.1       143.3
wind       215.3    132.3  61.4    39.1   120.1   171.6   293.3   619.4      4.4        78.6
solar      341.1     45.6  13.4   185.4   307.7   350.0   375.7   425.6      7.1       124.9
other_re    27.2      4.9  17.9     8.3    23.8    26.6    30.3    44.0      0.6        10.0

Total daily mean: 4837 MU/day
Total annual: 1770 TWh


## 2. The Volatility Story

**Coefficient of Variation (CV)** tells us which sources are steady vs wild. A CV of 10% means very stable; 50%+ means highly variable.

In [3]:
cv_data = pd.DataFrame({
    'fuel': fuel_labels,
    'cv': [df24[c].std() / df24[c].mean() * 100 for c in fuel_cols],
    'color': [palette[c] for c in fuel_cols],
}).sort_values('cv', ascending=True)

fig = go.Figure(go.Bar(
    y=cv_data['fuel'], x=cv_data['cv'],
    orientation='h',
    marker_color=cv_data['color'],
    text=[f'{v:.0f}%' for v in cv_data['cv']],
    textposition='outside',
))

fig.update_layout(
    title='Volatility by Fuel Type (Coefficient of Variation, 2024)',
    xaxis_title='CV (%)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=400, width=700,
)
fig.show()

print('\nInterpretation:')
print('  Nuclear is the metronome — almost constant output every day.')
print('  Coal is surprisingly steady too — it\'s the baseload backbone.')
print('  Wind is the wildcard — swings massively with the monsoon.')
print('  Hydro has huge seasonal range but follows a predictable pattern.')


Interpretation:
  Nuclear is the metronome — almost constant output every day.
  Coal is surprisingly steady too — it's the baseload backbone.
  Wind is the wildcard — swings massively with the monsoon.
  Hydro has huge seasonal range but follows a predictable pattern.


## 3. Distribution Shapes — Histograms

In [4]:
fig = make_subplots(rows=2, cols=4,
                    subplot_titles=fuel_labels,
                    vertical_spacing=0.15, horizontal_spacing=0.08)

for i, (col, label) in enumerate(zip(fuel_cols, fuel_labels)):
    row, colnum = (i // 4) + 1, (i % 4) + 1
    fig.add_trace(go.Histogram(
        x=df24[col].dropna(), nbinsx=30,
        marker_color=palette[col],
        name=label, showlegend=False,
        hovertemplate='%{x:.0f} MU: %{y} days<extra></extra>',
    ), row=row, col=colnum)
    fig.update_xaxes(title_text='MU/day', row=row, col=colnum, title_font_size=10)
    fig.update_yaxes(title_text='Days', row=row, col=colnum, title_font_size=10)

fig.update_layout(
    title='Distribution of Daily Generation by Fuel Type (2024)',
    height=550, plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
)
fig.show()

## 4. Box Plots — Side by Side (Normalized Scale)

In [5]:
fig = go.Figure()

for col, label in zip(fuel_cols, fuel_labels):
    fig.add_trace(go.Box(
        y=df24[col].dropna(),
        name=label,
        marker_color=palette[col],
        boxmean='sd',
        hovertemplate='%{y:.0f} MU<extra></extra>',
    ))

fig.update_layout(
    title='Daily Generation Distribution by Fuel Type (2024)',
    yaxis_title='Generation (MU/day)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
    showlegend=False,
)
fig.show()

## 5. The Pie — Annual Generation Share

In [6]:
annual = df24[fuel_cols].sum()

fig = go.Figure(go.Pie(
    labels=fuel_labels,
    values=annual.values,
    marker_colors=[palette[c] for c in fuel_cols],
    textinfo='label+percent',
    textposition='outside',
    hovertemplate='%{label}: %{value:.0f} GWh (%{percent})<extra></extra>',
    hole=0.3,
))

fig.update_layout(
    title=f'India Grid 2024 — Annual Generation Mix ({annual.sum()/1000:.0f} TWh total)',
    height=500, width=600,
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
)
fig.show()

## 6. Fossil vs Clean — The Big Split

In [7]:
fossil = df24[['coal', 'lignite', 'gas']].sum(axis=1)
clean = df24[['nuclear', 'hydro', 'wind', 'solar', 'other_re']].sum(axis=1)

print('Fossil (Coal + Lignite + Gas) vs Clean (Nuclear + Hydro + Wind + Solar + Other RE)')
print(f'  Fossil daily mean: {fossil.mean():.0f} MU ({fossil.mean()/df24["total"].mean()*100:.1f}%)')
print(f'  Clean daily mean:  {clean.mean():.0f} MU ({clean.mean()/df24["total"].mean()*100:.1f}%)')
print(f'  Fossil annual: {fossil.sum()/1000:.0f} TWh')
print(f'  Clean annual:  {clean.sum()/1000:.0f} TWh')
print(f'\n  Best clean day: {clean.max():.0f} MU ({(clean.max()/(fossil+clean).iloc[clean.argmax()]*100):.1f}% of total) on {df24.iloc[clean.argmax()]["date"].strftime("%B %d")}')
print(f'  Worst clean day: {clean.min():.0f} MU ({(clean.min()/(fossil+clean).iloc[clean.argmin()]*100):.1f}% of total) on {df24.iloc[clean.argmin()]["date"].strftime("%B %d")}')

Fossil (Coal + Lignite + Gas) vs Clean (Nuclear + Hydro + Wind + Solar + Other RE)
  Fossil daily mean: 3715 MU (76.5%)
  Clean daily mean:  1122 MU (23.1%)
  Fossil annual: 1360 TWh
  Clean annual:  411 TWh

  Best clean day: 1765 MU (37.1% of total) on August 26
  Worst clean day: 696 MU (15.9% of total) on February 04


## Key Findings

1. **Coal is king**: ~73% of generation. Mean ~3,529 MU/day. The grid breathes coal.
2. **Nuclear is the metronome**: Lowest CV (~8-10%), barely moves day to day. The steady heartbeat.
3. **Wind is the wildcard**: Highest CV (~60-70%), swings from near-zero in winter to 400+ in monsoon.
4. **Hydro has the biggest absolute swing**: From ~200 MU in winter to 700+ in monsoon — but follows a predictable seasonal arc.
5. **Solar is more consistent than you'd think**: Even in monsoon it stays above 190 MU. Growing steadily.
6. **Lignite and gas are minor players**: ~2% each. Gas is interesting because it's dispatchable (can ramp up/down fast).
7. **Fossil vs clean split**: Roughly 76% fossil, 24% clean. India's grid is still overwhelmingly thermal.

→ Next: Notebook 03 — Time Series deep dive